# Confidentiality-Aware Assistant — Stage 2 Evaluation

This notebook reproduces the baseline-vs-protected experiment and detector metrics on synthetic university policy data.

In [ ]:
from pathlib import Path
import sys, json, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from modules.detector import detect_sensitive_content
from modules.access_control import check_access
from modules.summarizer import generate_safe_summary
DOCS = pd.read_csv(ROOT / "data" / "documents.csv")
DOCS.head()

## 1. Dataset and policy
20 synthetic documents are evaluated against Student, Faculty and Admin roles.

In [ ]:
roles=["Student","Faculty","Admin"]
print("documents:",len(DOCS), "role-document cases:",len(DOCS)*len(roles))
print(DOCS["permission_label"].value_counts())

## 2. Detector examples
The detector combines configurable rules with local contextual similarity and optional spaCy patterns.

In [ ]:
examples=["employee compensation package is private", "staff appraisal discusses individual performance", "department spending records contain private financial figures", "library workshop closes Friday"]
for x in examples:
    print(x, "=>", detect_sensitive_content(x))

## 3. Quantitative results

In [ ]:
metrics=json.loads((ROOT/"experiments"/"metrics.json").read_text())
pd.DataFrame([metrics]).T.rename(columns={0:"value"})

## 4. Interpretation
The synthetic baseline leaks sensitive evidence in unauthorised first-sentence summaries, while the protected workflow blocks unauthorised summaries before content is returned. Detector precision/recall are measured separately because confidentiality detection can trade false positives for lower false negatives.

In [ ]:
results=pd.read_csv(ROOT/"experiments"/"experiment_results.csv")
results[["document_id","role","authorised","baseline_leak","protected_leak"]].head(15)